In [1]:
import logging
import os
import sys

from meridian.planner import adhoc_data_loader

from meridian.planner.adhoc_data_loader import AdhocDataLoader

In [2]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Excel file path (update this to match the actual location)
excel_file_path = (
  '/Users/mariappan.subramanian/Library/CloudStorage/'
  'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/mmm_input_artifacts.xlsx'
)

# Check if file exists
if not os.path.exists(excel_file_path):
  print(f"ERROR: Excel file not found at {excel_file_path}")
  print("Please update the excel_file_path variable to point to the correct location.")
  sys.exit(1)

# Model configuration based on actual Excel file structure
model_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_type': 'non_revenue',
  'kpi_col': 'conversions',
  'revenue_per_kpi_col': 'revenue_per_conversion',

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3'],

  # control inputs
  'control_cols': ['sentiment_score_control', 'competitor_activity_score_control']
  }

1. Create InputData

In [3]:
adhoc_data_loader = AdhocDataLoader(file_name=excel_file_path, model_config=model_config)
data = adhoc_data_loader.build_input_data()

INFO: Loaded Data sheet with shape: (3120, 17)
INFO: Loaded Coefficients sheet with shape: (20, 5)
INFO: Loaded Parameters sheet with shape: (4, 4)
INFO: Data validation passed - all required columns present
INFO: Successfully created InputData object


In [4]:
inference_data = adhoc_data_loader.get_inference_data()

INFO: Parameters validation passed - all requirements satisfied
INFO: Parameters reordered to match channel sequence: ['Channel0', 'Channel1', 'Channel2', 'Channel3']
INFO: Processed 3 media channel parameters
INFO: Processed 1 R&F channel parameters
INFO: Successfully processed parameters for 4 channels
INFO: Successfully created DataArrays for 6 parameter types
INFO: Successfully processed media parameters as DataArrays
INFO: Coefficients validation passed - all requirements satisfied
INFO: Successfully created coefficients DataArrays for 2 coefficient types
INFO: Successfully processed media coefficients as DataArrays
INFO: Data validation passed - all required columns present
INFO: Successfully created InputData object
INFO: Input validation passed for PointInferenceData creation
I0000 00:00:1756418101.848380 1656207 service.cc:148] XLA service 0x168a423d0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756418101.848410 1656207 s

In [7]:
from meridian.model import model
from meridian.model import spec

model_spec = spec.ModelSpec()
mmm = model.Meridian(input_data=data, model_spec=model_spec, inference_data=inference_data)
mmm.sample_prior(n_draws=100, seed=42)


In [9]:
from meridian.analysis import optimizer
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

In [10]:
optimization_results

OptimizationResults(meridian=<meridian.model.model.Meridian object at 0x16ae05910>, analyzer=<meridian.analysis.analyzer.Analyzer object at 0x16b792840>, spend_ratio=<xarray.DataArray 'non_optimized' (channel: 4)> Size: 32B
array([0.99963716, 0.99992993, 1.00146463, 0.99843775])
Coordinates:
  * channel  (channel) <U8 128B 'Channel0' 'Channel1' 'Channel2' 'Channel3', spend_bounds=(array([0.7]), array([1.3])), _nonoptimized_data=<xarray.Dataset> Size: 512B
Dimensions:              (channel: 4, metric: 4)
Coordinates:
  * channel              (channel) object 32B 'Channel0' ... 'Channel3'
  * metric               (metric) <U6 96B 'mean' 'median' 'ci_lo' 'ci_hi'
Data variables:
    spend                (channel) int64 32B 40400000 27600000 23300000 19600000
    pct_of_spend         (channel) float64 32B 0.3643 0.2489 0.2101 0.1767
    incremental_outcome  (channel, metric) float32 64B 6.518e+07 ... 9.881e+07
    effectiveness        (channel, metric) float32 64B 0.01794 ... 0.05929
    ro

In [8]:
mmm.inference_data = inference_data

AttributeError: property 'inference_data' of 'Meridian' object has no setter

In [ ]:
# param_arrays_dict = adhoc_data_loader.get_processed_parameter_arrays()
# coeff_arrays_dict = adhoc_data_loader.get_processed_coefficients_arrays()

INFO: Coefficients validation passed - all requirements satisfied
INFO: Successfully created coefficients DataArrays for 2 coefficient types
INFO: Successfully processed media coefficients as DataArrays


In [7]:
# from meridian.model import model
# from meridian.model import spec


# # Configure the model
# model_spec = spec.ModelSpec()
# mmm = model.Meridian(input_data=data, model_spec=model_spec)